# PaperScraper Anthropic API workflow

This notebook runs PaperScraper with the Anthropic API. It assumes you are already in the correct Python environment and that `ANTHROPIC_API_KEY` is available.

## 1. Set API keys

PaperScraper uses separate credentials for paper search/download and for model calls. You can either export keys in the notebook session or run the interactive `ps_*` config commands in a terminal before opening the notebook.

Search/download keys:

```bash
export ELSEVIER_API_KEY="..."      # Elsevier/Scopus search, Elsevier text, Elsevier PDFs
export CORE_API_KEY="..."          # CORE search and CORE PDFs
export UNPAYWALL_EMAIL="you@example.com"  # Unpaywall PDF lookup
```

Equivalent PaperScraper config commands:

```bash
ps_elsevier_key
ps_core_key
ps_unpaywall_email
```

Anthropic model key:

```bash
export ANTHROPIC_API_KEY="..."
```

Equivalent PaperScraper config command:

```bash
ps_anthropic_key
```

## 2. Configure Anthropic model profiles

PaperScraper keeps separate text and vision profiles. Use a Claude model that supports the inputs you plan to scrape.

In [ ]:
%%bash
ps_model_config text \
  --provider anthropic \
  --model claude-3-5-sonnet-latest

ps_model_config vision \
  --provider anthropic \
  --model claude-3-5-sonnet-latest

ps_model_status

## 3. Search for papers

This searches all configured search sources and writes a streamlined SQLite corpus to `papers.db`. Configure source credentials such as `ELSEVIER_API_KEY` and/or `CORE_API_KEY` before running this.

In [ ]:
%%bash
ps_search "Lithium solid electrolyte" papers.db \
  --source all \
  --count 10

## 4. Download paper content

`ps_download` uses every configured download source. Unpaywall uses `UNPAYWALL_EMAIL`, CORE uses `CORE_API_KEY`, and Elsevier uses `ELSEVIER_API_KEY`.

In [ ]:
%%bash
ps_download papers.db \
  --format both \
  --source all

## 5. Scrape structured data

This runs the bundled `sse` recipe over downloaded text and PDF images. Use `--mode text` if you only want text extraction.

In [ ]:
%%bash
ps_scrape papers.db sse \
  --mode text-images \
  --image-context paper-text

## 6. Store extracted rows

After reviewing `temp_scraped_materials.csv`, store the converted rows in `materials.csv`.

In [ ]:
%%bash
ps_store papers.db temp_scraped_materials.csv materials.csv sse --assume-yes